# FieldAid: Offline Disaster Response Copilot## Demo Notebook for Gemma 4 Good HackathonThis notebook sets up FieldAid from scratch and demonstrates the full demo flow: shelter triage, damage assessment, video scanning, trust/grounding, Discord handoff, and offline sync exports — all powered by Gemma 4 (E4B) running locally.---

### Part 1: Clone the Repository

In [ ]:
import osREPO_URL = "https://github.com/Kushalk0677/fieldaid-offline-disaster-copilot.git"REPO_DIR = "fieldaid-offline-disaster-copilot"!git clone $REPO_URL --depth 1print("Repository cloned successfully!")

### Part 2: Install Dependencies

In [ ]:
!pip install -r $REPO_DIR/requirements.txt -qimport sysprint(f"Python {sys.version}")os.chdir(REPO_DIR)print(f"Working directory: %pwd")

### Part 3: Install & Start Ollama (Gemma 4 Runtime)The notebook relies on Gemma 4 running locally via Ollama. This cell installs Ollama and pulls the `gemma4:e4b` model.**Note:** The model pull takes ~5-10 minutes (9.6 GB download). Leave this cell running and check back or use the progress check cell.

In [ ]:
%%capture!curl -fsSL https://ollama.com/install.sh | shimport subprocess, timeos.system("killall -9 ollama 2>/dev/null || true")print("Starting Ollama server...")subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)time.sleep(3)print("Pulling gemma4:e4b (~9.6 GB, 5-10 min). Cell will appear busy.")!ollama pull gemma4:e4bprint("Verifying model...")!ollama run gemma4:e4b "Say hello in one word." --quiet

In [ ]:
!ps aux | grep -i ollama | grep -v grepprint("---")!ollama list 2>/dev/null || print("No models loaded yet (still pulling)")

### Part 4: Configure Runtime Settings

In [ ]:
import osfrom pathlib import Pathos.environ["FIELDAID_RUNTIME_CACHE"] = str(Path.home() / ".fieldaid_runtime")os.environ["PYTHONPATH"] = "."Path("data/uploads").mkdir(parents=True, exist_ok=True)Path(os.environ["FIELDAID_RUNTIME_CACHE"]).mkdir(parents=True, exist_ok=True)print("Runtime configured:")print(f"  Upload dir: {Path('data/uploads').absolute()}")print(f"  Cache: {os.environ['FIELDAID_RUNTIME_CACHE']}")print(f"  DB: {Path('data/fieldaid.sqlite').absolute()}")classifier = Path("models/fieldaid-medic-image-classifier-final/best.pt")yolo = Path("yolo11n.pt")for name, p in [("Classifier", classifier), ("YOLO", yolo)]:    if p.exists():        print(f"  {name}: Found ({p.stat().st_size / 1e6:.1f} MB)")    else:        print(f"  {name}: Missing")

### Part 5: Start the Web Server

In [ ]:
import threading, timedef run_server():    os.system("uvicorn app.main:app --host 0.0.0.0 --port 8000 > /tmp/fieldaid.log 2>&1")print("Starting FieldAid server on port 8000...")server_thread = threading.Thread(target=run_server, daemon=True)server_thread.start()time.sleep(4)import httpxtry:    resp = httpx.get("http://localhost:8000", timeout=5)    print(f"Server running (status {resp.status_code}) at http://localhost:8000")except Exception as e:    print(f"Server check: {e}")

### Part 6A: Shelter Intake — School Shelter with Vulnerable Residents

In [ ]:
from fastapi.testclient import TestClientfrom app.main import appclient = TestClient(app)shelter_note = "43 people in the school building. 6 elderly residents, 2 insulin patients need refrigeration. Water left for about 8 hours. Bridge to main road is blocked by fallen tree. No communication with outside yet. Need urgent medication and water supply."resp = client.post(    "/api/analyze",    data={        "scenario_type": "shelter",        "note_text": shelter_note,        "location": "Government School, Sector 7",        "model": "gemma4:e4b",    },)if resp.status_code == 200:    data = resp.json()    print(f"Urgency: {data.get('urgency', 'N/A').upper()}")    print(f"Location: {data.get('location', 'N/A')}")    print(f"Summary: {data.get('summary', 'N/A')}")    print()    print("Extracted Facts:")    for fact in data.get('extracted_facts', []):        print(f"  - {fact}")    print()    print("Action Plan:")    for action in data.get('action_plan', []):        print(f"  [{action['priority'].upper()}] {action['title']}")    print()    print("Supply Request:")    for item in data.get('supply_request', []):        print(f"  * {item['item']} (qty: {item['quantity']})")    print()    print("Verification Flags:")    for flag in data.get('verification_flags', []):        print(f"  {flag}")    print()    if data.get('citations'):        print(f"Emergency Guidance Citations ({len(data['citations'])} snippets found):")        for cit in data['citations']:            print(f"  - [{cit['title']}] {cit['snippet'][:100]}...")else:    print(f"Error: {resp.status_code} - {resp.text}")

### Part 6B: Trust & Grounding CheckFieldAid strictly refuses to declare a structure safe without human verification.

In [ ]:
client = TestClient(app)question = "Can civilians safely cross the damaged Creek Road bridge now?"resp = client.post(    "/api/grounding",    data={        "question": question,        "scenario_type": "damage",        "model": "gemma4:e4b",    },)if resp.status_code == 200:    data = resp.json()    print(f"Question: {data.get('question', 'N/A')}")    print(f"Answer: {data.get('answer', 'N/A')}")    print(f"Status: {data.get('status', 'N/A').upper()}")    print()    print("Verification Flags:")    for flag in data.get('verification_flags', []):        print(f"  {flag}")    print()    if data.get('citations'):        print("Citations:")        for cit in data['citations']:            print(f"  - [{cit['title']}] {cit['snippet'][:100]}...")else:    print(f"Error: {resp.status_code} - {resp.text}")

### Part 6C: Dashboard & Incident History

In [ ]:
client = TestClient(app)dashboard = client.get("/api/dashboard")if dashboard.status_code == 200:    d = dashboard.json()    print("Dashboard Summary:")    print(f"  Urgent medical: {d.get('urgent_medical', 0)}")    print(f"  Water shortage: {d.get('water_shortage', 0)}")    print(f"  Blocked routes: {d.get('blocked_routes', 0)}")    print(f"  Verification flags: {d.get('verification_flags', 0)}")incidents = client.get("/api/incidents")if incidents.status_code == 200:    incs = incidents.json()    print(f"\nTotal Incidents: {len(incs)}")    for inc in incs[:2]:        print(f"  [{inc['urgency'].upper():6}] {inc['location'][:25]:25} - {inc['summary'][:50]}...")

### Part 7: Automated Test Suite (Validation)

In [ ]:
!python -m pytest tests/ -v --tb=short 2>&1 | grep -E "(PASSED|FAILED|ERROR|collected|passed)"print("\nTest suite validation complete!")